# MedXIR retrieval, step 1: encode the queries with Qwen3-Embedding-8B

Generated by `scripts/build_retrieval_notebook.py`. Edit the generator, not this file.

Loads only the model; no vector dataset is attached. Writes `query_vectors.npy` and `query_ids.json`.
Run on RTX Pro 6000 with Internet OFF. Attach `boinhbo/medxir-package-models` and `boinhbo/medxir-retrieval-inputs`.

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

## 1. Inputs and GPU check

In [ ]:
if FAILED is None:
    with stage('setup'):
        import glob
        import subprocess

        import numpy as np
        import pandas as pd
        import torch

        MODEL_DIR = 'Qwen3-Embedding-8B'
        TASK = 'Given a web search query, retrieve relevant passages that answer the query'
        MAX_LENGTH = 1024
        EMBED_BATCH = 8


        def one_match(pattern, description):
            hits = sorted(glob.glob(pattern, recursive=True))
            assert len(hits) == 1, f'{description}: expected one match, found {hits[:10]}'
            return Path(hits[0])


        QUERY_PATH = one_match('/kaggle/input/**/query.parquet', 'query.parquet')
        MODEL_PATH = one_match(f'/kaggle/input/**/{MODEL_DIR}/config.json', 'packaged Qwen model').parent
        print('query:', QUERY_PATH, '| model:', MODEL_PATH)

        print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                             capture_output=True, text=True).stdout)
        gpu_name = torch.cuda.get_device_name(0)
        assert '6000' in gpu_name, f'Expected RTX Pro 6000, got {gpu_name!r}'
        print('GPU:', gpu_name)

## 2. Encode the queries with the corpus instruction and pooling

In [ ]:
if FAILED is None:
    with stage('encode'):
        from transformers import AutoModel, AutoTokenizer
        import torch.nn.functional as F

        os.environ['HF_HUB_OFFLINE'] = '1'
        tok = AutoTokenizer.from_pretrained(str(MODEL_PATH))
        tok.padding_side = 'left'
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        model = AutoModel.from_pretrained(str(MODEL_PATH), torch_dtype=dtype,
                                          attn_implementation='sdpa').cuda().eval()

        queries = pd.read_parquet(QUERY_PATH, columns=['id', 'query'])
        assert len(queries) == 1200 and queries['id'].is_unique
        texts = [f'Instruct: {TASK}\nQuery:{q}' for q in queries['query'].astype(str)]


        def last_token_pool(hidden, mask):
            if mask[:, -1].sum() == mask.shape[0]:
                return hidden[:, -1]
            lengths = mask.sum(dim=1) - 1
            return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


        @torch.inference_mode()
        def encode(texts):
            tokenized = tok(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']
            order = np.argsort([-len(row) for row in tokenized], kind='stable')
            vectors = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
            for start in range(0, len(order), EMBED_BATCH):
                index = order[start:start + EMBED_BATCH]
                batch = tok.pad({'input_ids': [tokenized[i] for i in index]}, return_tensors='pt').to('cuda')
                hidden = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
                               use_cache=False).last_hidden_state
                pooled = F.normalize(last_token_pool(hidden, batch['attention_mask']).float(), p=2, dim=1)
                vectors[index] = pooled.cpu().numpy()
            return vectors


        query_vectors = encode(texts)
        assert query_vectors.shape == (1200, model.config.hidden_size)
        assert np.isfinite(query_vectors).all()
        del model, tok
        torch.cuda.empty_cache()
        np.save(OUT / 'query_vectors.npy', query_vectors)
        (OUT / 'query_ids.json').write_text(json.dumps(queries['id'].tolist()) + '\n', encoding='utf-8')
        print('encoded', len(queries), 'queries | elapsed', round(time.perf_counter() - T0, 1), 's')

## 3. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'stage_status.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')